# TF‑IDF + XGBoost (Generic) with Gensim Cleaning + Optional Lemmatization

*Generated: 2025-10-13T21:52:32*

**What you get:**
- Read **one or many CSVs** (local paths, globs like `data/*.csv`, or S3 URLs if `s3fs` is available)
- **Gensim**-based text cleaning (lower‑casing, accent/punct removal, stopwords) + **optional lemmatization**
- **TF‑IDF** features → **XGBoost** multiclass classifier
- Handles **imbalanced** labels via class weights
- **Evaluation**: classification report (with label names) + confusion matrix plot
- **Artifacts**: saves model, TF‑IDF vectorizer, label encoder, and metrics to `artifacts/`

> Tip: If you’re air‑gapped/egress‑blocked, keep lemmatization off or pre‑download NLTK corpora (`wordnet`, `omw-1.4`).


In [ ]:
import os, sys, json, math, glob, re, warnings, pathlib
import numpy as np
import pandas as pd
from typing import List

from datetime import datetime

warnings.filterwarnings('ignore')

# --- Optional: detect SageMaker
IS_SAGEMAKER = 'SM_CURRENT_HOST' in os.environ
print('Running in SageMaker?' , IS_SAGEMAKER)

# Core ML / NLP
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import remove_stopwords
try:
    # NLTK (optional) for lemmatization
    import nltk
    from nltk.stem import WordNetLemmatizer
    _NLTK_AVAILABLE = True
except Exception as e:
    print('[WARN] NLTK not available; lemmatization will be disabled unless installed.')
    _NLTK_AVAILABLE = False

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

try:
    import joblib
except Exception:
    from sklearn.externals import joblib  # fallback for older environments

print('Imports OK.')


In [ ]:
# -----------------------------
# Parameters (EDIT THESE)
# -----------------------------

# CSV inputs: can be a single path, a glob, or a list (e.g., ["data/part1.csv", "data/part2.csv"]).
CSV_INPUTS = ["sample.csv"]  # replace with your paths or globs; supports 's3://bucket/key' if s3fs is installed
CSV_HAS_HEADER = True         # set False if your CSVs have NO header row
CSV_SEP = ","                # e.g., ";" for semicolon‑separated files

# Columns
TEXT_COL = "text"
LABEL_COL = "label"

# Cleaning & lemmatization
APPLY_LEMMATIZATION = True    # Set False if you lack NLTK data or internet.
LOWERCASE_FIRST = True

# TF‑IDF settings
MAX_FEATURES = 50000
MIN_DF = 2
MAX_DF = 0.9
NGRAM_RANGE = (1, 2)

# Train/Val/Test split
TEST_SIZE = 0.10
VAL_SIZE = 0.10
RANDOM_STATE = 42

# Imbalance handling
USE_CLASS_WEIGHTS = True     # If True, compute balanced class weights

# XGBoost hyperparams (tune for your dataset)
XGB_PARAMS = dict(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective='multi:softprob',
    tree_method='hist',  # 'gpu_hist' if you have working GPU
    random_state=RANDOM_STATE,
)
EARLY_STOPPING_ROUNDS = 25

# Artifacts
ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

print('Parameters set.')


In [ ]:
# If lemmatization is enabled, attempt to verify/download NLTK resources.
if APPLY_LEMMATIZATION and _NLTK_AVAILABLE:
    try:
        nltk.data.find('corpora/wordnet')
        nltk.data.find('corpora/omw-1.4')
        print('NLTK corpora already present.')
    except LookupError:
        print('[INFO] NLTK corpora not found. Attempting download (requires internet)...')
        try:
            nltk.download('wordnet', quiet=True)
            nltk.download('omw-1.4', quiet=True)
            print('Downloaded wordnet + omw-1.4.')
        except Exception as e:
            print('[WARN] Could not download NLTK corpora. Proceeding without lemmatization.')
            APPLY_LEMMATIZATION = False
else:
    if APPLY_LEMMATIZATION and not _NLTK_AVAILABLE:
        print('[WARN] NLTK not available. Disabling lemmatization.')
        APPLY_LEMMATIZATION = False
    else:
        print('Lemmatization disabled by config.')


In [ ]:
# -----------------------------
# Gensim cleaning + optional lemmatization
# -----------------------------
lemmatizer = None
if APPLY_LEMMATIZATION and _NLTK_AVAILABLE:
    try:
        lemmatizer = WordNetLemmatizer()
    except Exception as e:
        print('[WARN] Failed to init WordNetLemmatizer; disabling lemmatization.')
        APPLY_LEMMATIZATION = False

def clean_and_normalize(text: str) -> str:
    if not isinstance(text, str):
        text = '' if text is None else str(text)
    if LOWERCASE_FIRST:
        text = text.lower()
    # remove stopwords early to reduce noise
    text = remove_stopwords(text)
    # tokenization + deacc (remove accents and punct)
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    if APPLY_LEMMATIZATION and lemmatizer is not None:
        # Lemmatize token‑wise; no POS tagging here for speed/portability
        tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
    return ' '.join(tokens)

print('Cleaning function ready. Lemmatization:', APPLY_LEMMATIZATION)


In [ ]:
# -----------------------------
# Data loading: local, glob, or s3:// (requires s3fs installed)
# -----------------------------
def expand_inputs(paths_or_globs: List[str]) -> List[str]:
    out = []
    for p in paths_or_globs:
        if p.startswith('s3://'):
            out.append(p)
        else:
            # expand globs
            matches = glob.glob(p)
            if matches:
                out.extend(matches)
            else:
                out.append(p)
    return out

files = expand_inputs(CSV_INPUTS)
if not files:
    raise FileNotFoundError('No input files resolved from CSV_INPUTS.')
print('Resolved files:', files)

dfs = []
read_kwargs = {}
if not CSV_HAS_HEADER:
    read_kwargs['header'] = None
    read_kwargs['names'] = [TEXT_COL, LABEL_COL]
if CSV_SEP:
    read_kwargs['sep'] = CSV_SEP

for fp in files:
    try:
        df_part = pd.read_csv(fp, **read_kwargs)
        dfs.append(df_part)
        print(f'Loaded: {fp}  shape={df_part.shape}')
    except Exception as e:
        # If s3 path and s3fs missing, give a helpful hint
        if fp.startswith('s3://'):
            print(f"[ERROR] Could not read {fp}. If you're using S3 paths, install s3fs (e.g., pip install s3fs). Error: {e}")
        else:
            raise

df = pd.concat(dfs, axis=0, ignore_index=True)
print('Combined shape:', df.shape)

# Basic checks
if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
    raise ValueError(f'Expected columns {TEXT_COL!r} and {LABEL_COL!r} in the CSV(s). Found: {list(df.columns)}')
df = df[[TEXT_COL, LABEL_COL]].dropna()
df = df[(df[TEXT_COL].astype(str).str.len() > 0) & (df[LABEL_COL].astype(str).str.len() > 0)]
print('After dropping NA/empty:', df.shape)

df.head()

In [ ]:
print('Cleaning text... this may take a while for large datasets.')
df['clean_text'] = df[TEXT_COL].apply(clean_and_normalize)
print('Example cleaned text:')
display(df[['clean_text', LABEL_COL]].head())

# Encode labels
le = LabelEncoder()
y_all = le.fit_transform(df[LABEL_COL].astype(str))
class_names = list(le.classes_)
num_classes = len(class_names)
print('Classes:', class_names)
print('Num classes:', num_classes)


In [ ]:
# Stratified train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'], y_all, test_size=(TEST_SIZE + VAL_SIZE),
    stratify=y_all, random_state=RANDOM_STATE
)
rel_val = VAL_SIZE / (TEST_SIZE + VAL_SIZE)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=(1.0 - rel_val),
    stratify=y_temp, random_state=RANDOM_STATE
)
print('Split sizes:', len(X_train), len(X_val), len(X_test))


In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
)
Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)
print('Vector shapes:', Xtr.shape, Xva.shape, Xte.shape)


In [ ]:
sample_weight_train = None
if USE_CLASS_WEIGHTS:
    classes_arr = np.unique(y_train)
    cw = compute_class_weight(class_weight='balanced', classes=classes_arr, y=y_train)
    class_weight_map = {int(c): float(w) for c, w in zip(classes_arr, cw)}
    print('Class weights:', class_weight_map)
    # Map per-sample weights for XGBoost
    sample_weight_train = np.array([class_weight_map[int(lbl)] for lbl in y_train])
else:
    print('Class weights disabled.')


In [ ]:
clf = XGBClassifier(**XGB_PARAMS)
eval_set = [(Xtr, y_train), (Xva, y_val)]
clf.fit(
    Xtr, y_train,
    sample_weight=sample_weight_train,
    eval_set=eval_set,
    verbose=False,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)
print('Best iteration:', getattr(clf, 'best_iteration', None))
print('Model trained.')


In [ ]:
def evaluate_split(X, y_true, split_name: str):
    y_prob = clf.predict_proba(X)
    y_pred = np.argmax(y_prob, axis=1)
    print(f'\n=== {split_name} Classification Report ===')
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    plt.figure(figsize=(6,6))
    disp.plot(values_format='d', xticks_rotation=45, colorbar=False)
    plt.title(f'Confusion Matrix — {split_name}')
    plt.tight_layout()
    plt.show()
    return cm

cm_val = evaluate_split(Xva, y_val, 'Validation')
cm_test = evaluate_split(Xte, y_test, 'Test')


In [ ]:
artifact_paths = {}
model_path = os.path.join(ART_DIR, 'xgb_model.joblib')
vec_path = os.path.join(ART_DIR, 'tfidf_vectorizer.joblib')
le_path = os.path.join(ART_DIR, 'label_encoder.joblib')
meta_path = os.path.join(ART_DIR, 'run_meta.json')

joblib.dump(clf, model_path)
joblib.dump(tfidf, vec_path)
joblib.dump(le, le_path)

meta = dict(
    timestamp=str(datetime.datetime.now()),
    params={
        'MAX_FEATURES': MAX_FEATURES,
        'MIN_DF': MIN_DF,
        'MAX_DF': MAX_DF,
        'NGRAM_RANGE': NGRAM_RANGE,
        'USE_CLASS_WEIGHTS': USE_CLASS_WEIGHTS,
        'XGB_PARAMS': XGB_PARAMS,
        'EARLY_STOPPING_ROUNDS': EARLY_STOPPING_ROUNDS,
        'APPLY_LEMMATIZATION': APPLY_LEMMATIZATION,
    },
    classes=class_names,
)
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

artifact_paths.update(dict(model=model_path, vectorizer=vec_path, label_encoder=le_path, meta=meta_path))
print('Saved artifacts:', json.dumps(artifact_paths, indent=2))


In [ ]:
N_SHOW = 10
y_prob_val = clf.predict_proba(Xva)
y_pred_val = np.argmax(y_prob_val, axis=1)

# Per-sample cross-entropy loss
sample_loss = -np.log(y_prob_val[np.arange(len(y_val)), y_val] + 1e-9)

df_val = pd.DataFrame({
    'text': list(X_val),
    'true_label_id': y_val,
    'pred_label_id': y_pred_val,
    'true_label': [class_names[i] for i in y_val],
    'pred_label': [class_names[i] for i in y_pred_val],
    'loss': sample_loss,
    'correct': y_pred_val == y_val,
})

#display(df_val.head(N_SHOW))

# Sort by descending loss (worst predictions first)
df_val_sorted = df_val.sort_values(by='loss', ascending=False)
display(df_val_sorted.head(N_SHOW))

# y_prob_val[np.arange(len(y_val)), y_val] extracts the predicted probability assigned to the true class for each sample.
# Taking the negative log gives the cross-entropy loss for that sample.
# Sorting by loss descending highlights examples the model is least confident or most wrong about.

In [ ]:
def predict_texts(texts: List[str]):
    # Clean -> vectorize -> predict
    cleaned = [clean_and_normalize(t) for t in texts]
    Xv = tfidf.transform(cleaned)
    y_prob = clf.predict_proba(Xv)
    y_pred = np.argmax(y_prob, axis=1)
    return [class_names[i] for i in y_pred]

predict_texts(["This is a wonderful day!", "The device failed to start and crashed twice."])

## Optional: Saving artifacts to S3
If you need to push artifacts to S3, ensure `boto3` credentials are configured, then run something like:
```python
import boto3
s3 = boto3.client('s3')
bucket = 'your-bucket'
prefix = 'tfidf-xgb-artifacts/'
for k, p in artifact_paths.items():
    s3.upload_file(p, bucket, prefix + os.path.basename(p))
```
For reading CSVs directly from S3, install `s3fs` and pass `s3://...` URLs in `CSV_INPUTS`.

## Notes on requirements
- **Gensim** ≥ 4.x
- **XGBoost** ≥ 2.x
- **scikit-learn** ≥ 1.3
- **NLTK** (optional for lemmatization; needs `wordnet` + `omw-1.4` corpora)
- **s3fs** (optional, for `s3://` CSV reads)

If you’re egress‑blocked, disable lemmatization or pre‑install NLTK corpora on the instance.